## RNN: Backpropagation Through Time (BPTT)

### 1. Introduction

Backpropagation Through Time (BPTT) for training Recurrent Neural Networks (RNNs). 
- Define forward and backward passes
- Derive gradients
- Explore the effect of shared weights across time
- Understand vanishing/exploding gradients

In [ ]:
import numpy as np

### 2. RNN Setup

In [2]:
# Define activation function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def dsigmoid(x):
    s = sigmoid(x)
    return s * (1 - s)

# Hyperparameters
input_size = 1
hidden_size = 2
output_size = 1
T = 3  # sequence length

# Weights (random initialization)
Wx = np.random.randn(hidden_size, input_size)   # input to hidden
Wh = np.random.randn(hidden_size, hidden_size)  # hidden to hidden
Wy = np.random.randn(output_size, hidden_size)  # hidden to output
bh = np.zeros((hidden_size, 1))  # hidden bias
by = np.zeros((output_size, 1))  # output bias

# Inputs and target
x_seq = [np.random.randn(input_size, 1) for _ in range(T)]
y_true = [np.random.randn(output_size, 1) for _ in range(T)]

### 3. Forward Pass

In [3]:
# Store activations and hidden states
h = [np.zeros((hidden_size, 1))]  # h[-1] = 0
y = []

# Forward pass through time
for t in range(T):
    h_t = np.tanh(Wx @ x_seq[t] + Wh @ h[-1] + bh)
    y_t = Wy @ h_t + by
    h.append(h_t)
    y.append(y_t)

### 4. Loss Function (Mean Squared Error)

In [4]:
def mse_loss(y_pred, y_true):
    return 0.5 * np.sum((y_pred - y_true)**2)

loss = sum(mse_loss(y[t], y_true[t]) for t in range(T))
print("Loss:", loss)

Loss: 5.2227466356544845


### 5. Backpropagation Through Time (BPTT)

In [5]:
# Gradients
dWx = np.zeros_like(Wx)
dWh = np.zeros_like(Wh)
dWy = np.zeros_like(Wy)
dbh = np.zeros_like(bh)
dby = np.zeros_like(by)
dh_next = np.zeros((hidden_size, 1))

# Backward through time
for t in reversed(range(T)):
    dy = y[t] - y_true[t]
    dWy += dy @ h[t+1].T
    dby += dy

    dh = Wy.T @ dy + dh_next
    dtanh = (1 - h[t+1]**2) * dh

    dWx += dtanh @ x_seq[t].T
    dWh += dtanh @ h[t].T
    dbh += dtanh

    dh_next = Wh.T @ dtanh


### 6. Summary of Key Equations

In [6]:
"""
Forward Pass:
    h_t = tanh(Wx * x_t + Wh * h_{t-1} + bh)
    y_t = Wy * h_t + by

Backward Pass:
    ∂L/∂Wy += ∂L/∂y_t * h_t.T
    ∂L/∂Wh += ∂L/∂h_t * h_{t-1}.T
    ∂L/∂Wx += ∂L/∂h_t * x_t.T

Where:
    ∂L/∂h_t includes contributions from future time steps (via dh_next)
    ∂L/∂Wx, ∂L/∂Wh accumulate over all time steps
"""

print("Gradient shapes:")
print("dWx:", dWx.shape)
print("dWh:", dWh.shape)
print("dWy:", dWy.shape)

Gradient shapes:
dWx: (2, 1)
dWh: (2, 2)
dWy: (1, 2)


### 7. Gradient Flow Issues: Vanishing & Exploding Gradients

In [ ]:
"""
Observation:
- RNNs apply the same weights (Wh) across time.
- Gradients are products of many derivatives of tanh/sigmoid, which are < 1.
    => Causes vanishing gradients for long sequences.
- Alternatively, large weights can cause exploding gradients.

Solutions (not implemented here):
- Gradient clipping
- Using ReLU or LeakyReLU
- Using LSTM/GRU instead of vanilla RNN
"""

### 8. Comparison with Other Training Techniques
Compare BPTT with other training strategies for RNNs.

Key Comparisons:
- Full BPTT vs. Truncated BPTT
- Truncation Strategies (Fixed vs. Randomized)
- BPTT vs. Real-Time Recurrent Learning (RTRL)
- BPTT in GRUs and LSTMs
- BPTT vs. Gradient-Free Methods (e.g., Evolutionary Algorithms)

#### 8.1 Full BPTT vs. Truncated BPTT
Full BPTT:
- Computes gradients over the entire sequence.
- More accurate, but computationally expensive.
- Impractical for very long sequences.

Truncated BPTT:
- Only backpropagates errors for a fixed number of time steps (e.g., 5 or 10).
- Reduces memory and computational load.
- May lose long-range dependencies.
Example:
Instead of backpropagating over T = 100 steps, should do it over T_trunc = 10.

In [7]:
T_trunc = 2  # truncated steps

# Same forward pass...
# Modify BPTT loop:
for t in reversed(range(T)):
    if t < T - T_trunc:
        break  # stop early for truncation

#### 8.2 Randomized Truncation vs. Fixed Truncation
Fixed Truncation:
- Always backprop over a fixed window (e.g., 10 steps).

Randomized Truncation:
- Randomly choose the truncation window per batch or iteration.
- Introduces stochasticity that may help generalization.

This is sometimes called "stochastic truncated BPTT".

In [8]:
import random

T_trunc_random = random.randint(1, T)

print(f"Randomized truncation window: {T_trunc_random}")

Randomized truncation window: 3


#### 8.3 BPTT vs. Real-Time Recurrent Learning (RTRL)
RTRL:
- Online algorithm: updates weights at each time step.
- Keeps track of all partial derivatives in real-time.
- Accurate but scales poorly: O(n^4) for n hidden units!

BPTT:
- More efficient in practice (O(n^2)), but delayed updates (after full/truncated sequence).
- BPTT is preferred for deep learning frameworks due to efficiency.

Note:
    RTRL is sometimes used in biologically plausible learning or online systems.

#### 8.4 BPTT in GRUs and LSTMs
BPTT applies to GRU/LSTM the same way as standard RNNs:
- Still unroll the network across time.
- Gradients are computed using the chain rule.

The difference lies in internal gating mechanisms:
- GRU and LSTM mitigate vanishing gradients via additive updates (e.g., cell state in LSTM).
- This makes long-term dependencies easier to learn.

So while BPTT is the same algorithmically, GRU/LSTM are more robust in training.

#### 8.5 BPTT vs. Gradient-Free Methods
Gradient-Free Methods:
- E.g., Evolutionary Algorithms, Genetic Algorithms, Reinforcement Learning.
- Useful when gradients are hard to compute (non-differentiable models or noisy environments).

Pros:
- Can explore novel solutions and architectures.
- No gradient calculation needed.

Cons:
- Often less sample-efficient.
- Slower to converge.

In RNNs:
- Used for neuromorphic computing or evolving controllers.
- BPTT remains the most common method in deep learning due to efficiency and maturity.